In [32]:
#Problema de multiclass classification pt RNN (LSTM) cu word 2 vec

PASUL 1 - IMPORTS

In [33]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
import torch 
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import re
from sklearn.preprocessing import LabelEncoder
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split

PASUL 2 - VARIABILE GLOBALE

In [34]:
BATCH_SIZE = 32
EPOCHS = 30
LR = 1E-3 
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

PASUL 3 - CITIRE FISERE DATE

In [35]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

PASUL 4 - PREPROCESARE TEXT

In [36]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

train_df['text'] = train_df['text'].apply(preprocess_text)
test_df['text'] = test_df['text'].apply(preprocess_text)

PASUL 5 - WORD 2VEC

In [37]:
train_df['tokens'] = train_df['text'].apply(lambda x: x.split())
test_df['tokens'] = test_df['text'].apply(lambda x: x.split())

sentences = train_df['tokens'].to_list()

w2v = Word2Vec(
    sentences, 
    vector_size=200,
    window=7,
    min_count=3,
    epochs=10
)

PASUL 6 - VOCAB + ENCODING

In [38]:
vocab = {word: i+2 for i , word in enumerate(w2v.wv.index_to_key)}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1
embedding_matrix = np.zeros((len(vocab),200))
for word, i in vocab.items(): 
    if word in w2v.wv: 
        embedding_matrix[i] =w2v.wv[word]

def encode(tokens):
    return [vocab.get(w,1) for w in tokens]

train_df['encoded'] = train_df['tokens'].apply(encode)
test_df['encoded'] = test_df['tokens'].apply(encode)

PASUL 7 - PADDING

In [39]:
MAX_LEN = 200

def pad(seq):
    if len(seq) < MAX_LEN: 
        return seq[:MAX_LEN] + [0]*(MAX_LEN-len(seq))  
    else: 
        return seq[:MAX_LEN]

train_df['padded'] = train_df['encoded'].apply(pad)
test_df['padded'] = test_df['encoded'].apply(pad)

PASUL 8 - LABEL ENCODING

In [40]:
le = LabelEncoder()

train_df['label_enc'] = le.fit_transform(train_df['label'])

PASUL 9 - DATASET, DATALOADER

In [41]:
X = np.array(train_df['padded'].tolist())
y = np.array(train_df['label_enc'].tolist())

X_tr, X_val, y_tr, y_val = train_test_split(X,y, test_size =0.2 ,random_state = 42)

class TextDataset(Dataset):
    def __init__(self,X,y = None,test = False):
        self.X = torch.tensor(X, dtype = torch.long)
        if test == False:
            self.y = torch.tensor(y, dtype = torch.long)
        else:
            self.y = None

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self,idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        else:
            return self.X[idx]


train_dts = TextDataset(X_tr,y_tr, test = False)
val_dts = TextDataset(X_val,y_val, test = False)

train_loader = DataLoader(train_dts,batch_size = 64, shuffle = True)
val_loader = DataLoader(val_dts, batch_size = 64)


PASUL 10 - MODEL LSTM 

In [42]:
class LSTMModel(nn.Module):
    def __init__(self,embedding_matrix, hidden_dim, num_classes):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix,dtype = torch.float32),
            freeze = False
        )
        self.lstm = nn.LSTM(200, hidden_dim, batch_first = True,bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim*2, num_classes)

    def forward(self,x):
        x =self.embedding(x)
        out, _ = self.lstm(x)
        out = torch.mean(out, dim=1)
        out = self.dropout(out)
        out = self.fc(out)
        return out


model = LSTMModel(embedding_matrix,
                  hidden_dim = 128,
                  num_classes = len(le.classes_)).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr= 1e-3)

PASUL 11 - TRAINING LOOP 

In [43]:
for epoch in range(30):
    model.train()

    for X_batch, y_batch in tqdm(train_loader):
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f'Epoch: {epoch+1} / {EPOCHS}, Loss: {loss.item():.4f}')

100%|██████████| 200/200 [01:01<00:00,  3.27it/s]


Epoch: 1 / 30, Loss: 1.3754


100%|██████████| 200/200 [01:16<00:00,  2.61it/s]


Epoch: 2 / 30, Loss: 1.1657


100%|██████████| 200/200 [01:22<00:00,  2.43it/s]


...

Epoch: 28 / 30, Loss: 0.0346


100%|██████████| 200/200 [01:07<00:00,  2.97it/s]


Epoch: 29 / 30, Loss: 0.0069


100%|██████████| 200/200 [01:21<00:00,  2.46it/s]

Epoch: 30 / 30, Loss: 0.0907


In [44]:
model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in tqdm(val_loader):
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        
        outputs = model(X_batch)
        preds = torch.argmax(outputs, dim=1)  # pentru multi-clasă
        
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y_batch.cpu().numpy())

f1 = f1_score(all_targets, all_preds, average='macro')
print("F1 Macro:", f1)

100%|██████████| 50/50 [00:06<00:00,  7.64it/s]

F1 Macro: 0.8642046815238914


PASUL 12 - PREDICT + SUBMISSION

In [45]:
test_X = np.array(test_df['padded'].tolist())
test_dts = TextDataset(test_X, test = True)

test_loader = DataLoader(test_dts, batch_size = 64)

model.eval()
preds = []

with torch.no_grad():
    for X_batch in tqdm(test_loader):
        X_batch = X_batch.to(DEVICE)
        outputs = model(X_batch)
        pred= torch.argmax(outputs, dim = 1)
        preds.extend(pred.cpu().numpy())

pred_labels = le.inverse_transform(preds)

submission = pd.DataFrame({
    "SampleID": test_df["SampleID"],
    "label": pred_labels
})

submission.to_csv("submission_5_apr1st.csv", index=False)

100%|██████████| 32/32 [00:03<00:00,  9.34it/s]
